**Install**

In [1]:
!pip install -q pymupdf sentence-transformers faiss-cpu groq tqdm

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.9/24.9 MB 28.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 31.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 138.3/138.3 kB 7.5 MB/s eta 0:00:00


**Upload pdf**

In [2]:
from google.colab import files

print("Upload Swiggy Annual Report PDF...")
uploaded = files.upload()
PDF_PATH = list(uploaded.keys())[0]
print(f" Uploaded: {PDF_PATH}")

Upload Swiggy Annual Report PDF...


Saving Swiggy-Annual-Report-FY-2024-25.pdf to Swiggy-Annual-Report-FY-2024-25.pdf
 Uploaded: Swiggy-Annual-Report-FY-2024-25.pdf


**Config**

In [10]:
import os
from getpass import getpass

CHUNK_SIZE           = 400
CHUNK_OVERLAP        = 100
EMBED_MODEL          = "BAAI/bge-small-en-v1.5"
TOP_K                = 4
SIMILARITY_THRESHOLD = 0.45

os.environ["GROQ_API_KEY"] = getpass("Enter Groq API Key: ")
GROQ_MODEL = "llama-3.3-70b-versatile"

print(" Config ready")

Enter Groq API Key: ··········
 Config ready


**Ingest and chunks**

In [11]:
import re
import fitz
from dataclasses import dataclass
from typing import List

@dataclass
class Chunk:
    id:    int
    text:  str
    pages: List[int]


def clean_text(text: str) -> str:
    text = re.sub(r"[ \t]+", " ", text)
    text = re.sub(r"\n{3,}", "\n\n", text)
    text = re.sub(r"[^\x20-\x7E\n]", " ", text)
    return text.strip()


def load_pdf(path: str) -> List[dict]:
    doc   = fitz.open(path)
    pages = []
    for i, page in enumerate(doc, start=1):
        text = clean_text(page.get_text("text"))
        if text.strip():
            pages.append({"page": i, "text": text})
    doc.close()
    print(f" Extracted {len(pages)} pages")
    return pages


def chunk_pages(pages: List[dict]) -> List[Chunk]:
    # Flatten ALL words across ALL pages — cross-page chunking
    word_page = []
    for p in pages:
        for word in p["text"].split():
            word_page.append((word, p["page"]))

    chunks, step = [], CHUNK_SIZE - CHUNK_OVERLAP
    for start in range(0, len(word_page), step):
        window = word_page[start : start + CHUNK_SIZE]
        if not window:
            break
        text          = " ".join(w for w, _ in window)
        pages_covered = sorted({pg for _, pg in window})
        chunks.append(Chunk(id=len(chunks), text=text, pages=pages_covered))

    print(f" Created {len(chunks)} cross-page chunks")
    return chunks


pages  = load_pdf(PDF_PATH)
chunks = chunk_pages(pages)
print(f" Total pages : {len(pages)}")
print(f" Total chunks: {len(chunks)}")

 Extracted 178 pages
 Created 535 cross-page chunks
 Total pages : 178
 Total chunks: 535


**Embeddings & FAISS**

In [12]:
import numpy as np
import faiss
from sentence_transformers import SentenceTransformer

print(f"Loading embedding model: {EMBED_MODEL} ...")
embed_model = SentenceTransformer(EMBED_MODEL)

texts = [c.text for c in chunks]
print(f"Embedding {len(texts)} chunks — takes 3-5 min for 350 pages ...")

embeddings = embed_model.encode(
    texts,
    batch_size           = 64,
    show_progress_bar    = True,
    normalize_embeddings = True,
    convert_to_numpy     = True,
).astype("float32")

dim   = embeddings.shape[1]
index = faiss.IndexFlatIP(dim)
index.add(embeddings)

print(f" FAISS index ready — {index.ntotal} vectors, dim={dim}")

Loading embedding model: BAAI/bge-small-en-v1.5 ...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: BAAI/bge-small-en-v1.5
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Embedding 535 chunks — takes 3-5 min for 350 pages ...


Batches:   0%|          | 0/9 [00:00<?, ?it/s]

 FAISS index ready — 535 vectors, dim=384


**Retriever**

In [13]:
from dataclasses import dataclass

@dataclass
class RetrievedChunk:
    chunk: Chunk
    score: float


QUERY_EXPANSIONS = {
    "revenue":    "total revenue income from operations financial statement FY 2023-24",
    "loss":       "net loss profit loss after tax financial statement",
    "profit":     "net profit loss after tax financial statement",
    "cities":     "cities operations geographic presence locations",
    "instamart":  "instamart quick commerce instant delivery grocery",
    "board":      "board of directors members management",
    "ceo":        "CEO managing director executive leadership",
    "risk":       "risk factors challenges threats business risks regulatory",
    "csr":        "CSR corporate social responsibility sustainability community",
    "gov":        "gross order value GOV food delivery platform",
    "ebitda":     "EBITDA adjusted margin profitability",
    "employee":   "employees headcount workforce team",
    "swiggy one": "swiggy one membership subscription benefits free delivery",
    "membership": "swiggy one membership subscription benefits",
    "audit":      "audit committee auditor report financial",
    "dividend":   "dividend shareholders equity",
    "subsidiary": "subsidiary companies associates joint venture",
}

FINANCIAL_TERMS = [
    "total income", "revenue", "profit", "loss",
    "standalone", "consolidated", "financial statement",
    "balance sheet", "cash flow", "ebitda", "gross order"
]


def expand_query(query: str) -> str:
    q_lower = query.lower()
    extras  = [exp for kw, exp in QUERY_EXPANSIONS.items() if kw in q_lower]
    return query + " " + " ".join(extras) if extras else query


def embed_query(query: str) -> np.ndarray:
    return embed_model.encode(
        [expand_query(query)],
        normalize_embeddings = True,
        convert_to_numpy     = True,
    ).astype("float32")


def retrieve(query: str, top_k: int = TOP_K) -> List[RetrievedChunk]:
    q_emb       = embed_query(query)
    scores, ids = index.search(q_emb, top_k * 2)  # fetch 2x then re-rank

    query_words = set(query.lower().split())
    results     = []

    for score, idx in zip(scores[0], ids[0]):
        if idx < 0:
            continue

        chunk       = chunks[idx]
        text_lower  = chunk.text.lower()
        chunk_words = set(text_lower.split())

        # Boost 1: financial terms present
        fin_boost = 0.05 if any(t in text_lower for t in FINANCIAL_TERMS) else 0

        # Boost 2: query keyword overlap
        overlap       = len(query_words & chunk_words) / max(len(query_words), 1)
        keyword_boost = overlap * 0.08

        final_score = float(score) + fin_boost + keyword_boost

        if final_score >= SIMILARITY_THRESHOLD:
            results.append(RetrievedChunk(chunk=chunk, score=final_score))

    results.sort(key=lambda x: x.score, reverse=True)
    return results[:top_k]


def format_context(results: List[RetrievedChunk]) -> str:
    if not results:
        return "No relevant excerpts found."
    parts = []
    for i, r in enumerate(results, 1):
        pages = ", ".join(str(p) for p in r.chunk.pages)
        parts.append(
            f"--- Excerpt {i} | Page {pages} | score: {r.score:.3f} ---\n"
            f"{r.chunk.text}"
        )
    return "\n\n".join(parts)


print(" Retriever ready")

 Retriever ready


**Generator**

In [14]:
from groq import Groq

SYSTEM_PROMPT = """You are a precise Q&A assistant for the Swiggy Annual Report FY 2023-24.

STRICT RULES:
1. Answer ONLY using the provided context. Never use external knowledge.
2. If the answer is not in the context, say exactly:
   "I could not find this information in the Swiggy Annual Report."
3. Always cite page number(s), e.g. (Page 6).
4. If multiple pages support the answer, cite all of them.
5. Include exact numbers with units as shown in context.
6. For lists (e.g. board members), include all names mentioned in context.
7. Give the direct answer with page citation — no preamble."""

groq_client = Groq()


def generate_answer(query: str, context: List[RetrievedChunk]) -> str:
    if not context:
        return "I could not find this information in the Swiggy Annual Report."

    ctx_text = format_context(context)

    financial_words = ["revenue", "income", "profit", "loss", "ebitda", "gov"]
    if any(w in query.lower() for w in financial_words):
        if not any(ch.isdigit() for ch in ctx_text):
            return "I could not find this information in the Swiggy Annual Report."

    resp = groq_client.chat.completions.create(
        model       = GROQ_MODEL,
        messages    = [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user",   "content": f"Context:\n\n{ctx_text}\n\nQuestion: {query}"},
        ],
        temperature = 0,
        max_tokens  = 800,
    )
    return resp.choices[0].message.content.strip()


print("Generator ready")

Generator ready


**Ask Function**

In [15]:
def ask(query: str, show_context: bool = True) -> str:
    results = retrieve(query)
    answer  = generate_answer(query, results)

    print("\n" + "═" * 70)
    print(f"Q: {query}")
    print("═" * 70)
    print(f"\nA: {answer}\n")

    if show_context:
        if not results:
            print("    No chunks passed similarity threshold.")
        else:
            print("─" * 70)
            print("Supporting Context (top 3):")
            for i, r in enumerate(results[:3], 1):
                pages = ", ".join(str(p) for p in r.chunk.pages)
                print(f"\n  [{i}] Page {pages}  (score: {r.score:.4f})")
                print(f"      {r.chunk.text[:250].strip()} ...")
        print("─" * 70)

    return answer


print(" Ask ready")

 Ask ready


**Run All Questions**

In [16]:
import time

QUESTIONS = [
    "What was Swiggy's total revenue for FY 2023-24?",
    "What was Swiggy's net loss for FY 2023-24?",
    "What was the Gross Order Value for food delivery?",
    "How many cities does Swiggy operate in?",
    "What is Instamart and how did it perform?",
    "Who are the members of Swiggy's Board of Directors?",
    "Who is the CEO of Swiggy?",
    "What is Swiggy One membership?",
    "What are the key risk factors mentioned in the report?",
    "What are Swiggy's CSR initiatives?",
    "What is Swiggy's stock price today?",   # hallucination test
]

answers = {}

for q in QUESTIONS:
    print(f"\n⏳ {q[:60]}...")
    ans        = ask(q, show_context=False)
    answers[q] = ans
    print(f" {ans[:120]}")
    time.sleep(20)

print(f"\n Collected {len(answers)} answers")


⏳ What was Swiggy's total revenue for FY 2023-24?...

══════════════════════════════════════════════════════════════════════
Q: What was Swiggy's total revenue for FY 2023-24?
══════════════════════════════════════════════════════════════════════

A: 94,299.37 million (Page 145).

 94,299.37 million (Page 145).

⏳ What was Swiggy's net loss for FY 2023-24?...

══════════════════════════════════════════════════════════════════════
Q: What was Swiggy's net loss for FY 2023-24?
══════════════════════════════════════════════════════════════════════

A: Swiggy's net loss for FY 2023-24 was ` 25,416.71 million (Page 41) and ` 31,167.99 million on a consolidated basis (Page 104).

 Swiggy's net loss for FY 2023-24 was ` 25,416.71 million (Page 41) and ` 31,167.99 million on a consolidated basis (Page

⏳ What was the Gross Order Value for food delivery?...

══════════════════════════════════════════════════════════════════════
Q: What was the Gross Order Value for food delivery?
═════════════

**Accuracy Evaluation**

In [17]:
GROUND_TRUTH = {
    "What was Swiggy's total revenue for FY 2023-24?": [
        "116,343", "116", "total income", "revenue"
    ],
    "What was Swiggy's net loss for FY 2023-24?": [
        "23,502", "23", "loss", "net"
    ],
    "What was the Gross Order Value for food delivery?": [
        "order value", "gov", "gross", "delivery", "food"
    ],
    "How many cities does Swiggy operate in?": [
        "653", "cities", "operate"
    ],
    "What is Instamart and how did it perform?": [
        "instamart", "quick commerce", "grocery"
    ],
    "Who are the members of Swiggy's Board of Directors?": [
        "sriharsha", "anand", "kripalu", "director"
    ],
    "Who is the CEO of Swiggy?": [
        "sriharsha", "majety", "ceo", "managing director"
    ],
    "What is Swiggy One membership?": [
        "swiggy one", "membership", "subscription", "benefits"
    ],
    "What are the key risk factors mentioned in the report?": [
        "risk", "competition", "regulatory", "market", "challenge"
    ],
    "What are Swiggy's CSR initiatives?": [
        "csr", "social", "community", "education", "responsibility"
    ],
    "What is Swiggy's stock price today?": [
        "could not find", "not find", "not available"
    ],
}


def evaluate_accuracy(answers: dict, ground_truth: dict) -> dict:
    correct = 0
    total   = len(ground_truth)

    print("\n" + "═" * 70)
    print("            ANSWER-BY-ANSWER DEBUG")
    print("═" * 70)

    results = []
    for question, keywords in ground_truth.items():
        if question not in answers:
            print(f"\n? NOT RUN: {question[:60]}")
            results.append({"question": question, "pass": False})
            continue

        ans_lower = answers[question].lower()
        passed    = any(kw.lower() in ans_lower for kw in keywords)
        correct  += passed

        status = "correct" if passed else "wrong"
        print(f"\n{status} Q: {question[:60]}")
        print(f"   A: {answers[question][:120]}")
        if not passed:
            print(f"    Expected one of: {keywords}")

        results.append({"question": question, "pass": passed})

    accuracy = (correct / total) * 100

    print("\n" + "═" * 70)
    print("              ACCURACY EVALUATION REPORT")
    print("═" * 70)
    print(f"{'#':<3} {'STATUS':<8} {'QUESTION'}")
    print("─" * 70)
    for i, r in enumerate(results, 1):
        status = "PASS" if r["pass"] else " FAIL"
        print(f"{i:<3} {status:<8} {r['question'][:55]}")

    print("─" * 70)
    print(f"\n Score    : {correct} / {total}")
    print(f"Accuracy : {accuracy:.1f}%")

    if accuracy >= 90:
        grade = " Excellent"
    elif accuracy >= 70:
        grade = " Good"
    elif accuracy >= 50:
        grade = " Needs improvement"
    else:
        grade = " Poor"

    print(f" Grade    : {grade}")
    print("═" * 70)

    return {"score": correct, "total": total, "accuracy": accuracy, "results": results}


eval_results = evaluate_accuracy(answers, GROUND_TRUTH)


══════════════════════════════════════════════════════════════════════
            ANSWER-BY-ANSWER DEBUG
══════════════════════════════════════════════════════════════════════

wrong Q: What was Swiggy's total revenue for FY 2023-24?
   A: 94,299.37 million (Page 145).
    Expected one of: ['116,343', '116', 'total income', 'revenue']

correct Q: What was Swiggy's net loss for FY 2023-24?
   A: Swiggy's net loss for FY 2023-24 was ` 25,416.71 million (Page 41) and ` 31,167.99 million on a consolidated basis (Page

correct Q: What was the Gross Order Value for food delivery?
   A: Gross Order Value (GOV) for Food Delivery is described as the total monetary value of completed Food Delivery orders (gr

correct Q: How many cities does Swiggy operate in?
   A: Swiggy operates in 700+ cities across India for its food delivery service (Page 32) and has expanded its Quick Commerce 

correct Q: What is Instamart and how did it perform?
   A: Instamart is Swiggy's quick-commerce service, offer

**Retrival Quality**

In [18]:
RETRIEVAL_GT = {
    "What was Swiggy's total revenue for FY 2023-24?":     ["116,343", "total income"],
    "What was Swiggy's net loss for FY 2023-24?":          ["net loss", "profit/(loss)"],
    "Who are the members of Swiggy's Board of Directors?": ["sriharsha", "anand kripalu"],
    "Who is the CEO of Swiggy?":                           ["sriharsha majety", "managing director"],
    "What is Instamart and how did it perform?":           ["instamart", "quick commerce"],
    "What are Swiggy's CSR initiatives?":                  ["csr", "social responsibility"],
    "What are the key risk factors mentioned in the report?": ["risk", "competition"],
    "What is Swiggy One membership?":                      ["swiggy one", "membership"],
}


def evaluate_retrieval(ground_truth: dict) -> dict:
    hit_count = 0
    rr_sum    = 0.0
    total     = len(ground_truth)

    print("\n" + "═" * 70)
    print("           RETRIEVAL QUALITY REPORT")
    print("═" * 70)
    print(f"{'#':<3} {'HIT':<5} {'RANK':<6} {'QUESTION'}")
    print("─" * 70)

    for i, (question, keywords) in enumerate(ground_truth.items(), 1):
        retrieved    = retrieve(question)
        hit, rank    = False, None

        for j, r in enumerate(retrieved, 1):
            if any(kw.lower() in r.chunk.text.lower() for kw in keywords):
                hit, rank = True, j
                break

        if hit:
            hit_count += 1
            rr_sum    += 1.0 / rank

        status   = "correct" if hit else "wrong"
        rank_str = str(rank) if rank else "—"
        print(f"{i:<3} {status:<5} {rank_str:<6} {question[:55]}")

    hit_rate = (hit_count / total) * 100
    mrr      = rr_sum / total

    print("─" * 70)
    print(f"\n Hit Rate : {hit_rate:.1f}%  ({hit_count}/{total} found in top-{TOP_K})")
    print(f" MRR      : {mrr:.3f}  (1.0 = always rank #1)")
    print("═" * 70)

    return {"hit_rate": hit_rate, "mrr": mrr}


retrieval_results = evaluate_retrieval(RETRIEVAL_GT)


══════════════════════════════════════════════════════════════════════
           RETRIEVAL QUALITY REPORT
══════════════════════════════════════════════════════════════════════
#   HIT   RANK   QUESTION
──────────────────────────────────────────────────────────────────────
1   correct 2      What was Swiggy's total revenue for FY 2023-24?
2   wrong —      What was Swiggy's net loss for FY 2023-24?
3   correct 1      Who are the members of Swiggy's Board of Directors?
4   correct 1      Who is the CEO of Swiggy?
5   correct 1      What is Instamart and how did it perform?
6   correct 1      What are Swiggy's CSR initiatives?
7   correct 1      What are the key risk factors mentioned in the report?
8   correct 1      What is Swiggy One membership?
──────────────────────────────────────────────────────────────────────

 Hit Rate : 87.5%  (7/8 found in top-4)
 MRR      : 0.812  (1.0 = always rank #1)
══════════════════════════════════════════════════════════════════════


**Unseen data test**

In [19]:
TEST_QUESTIONS = [
    "What was the total expenditure for FY 2023-24?",
    "Who is the Chief Financial Officer of Swiggy?",
    "What was the depreciation amount for FY 2023-24?",
    "When was Swiggy incorporated?",
    "Who are the independent directors on the board?",
    "What was other income for FY 2023-24?",
    "What is Swiggy's Dineout business?",
    "How many employees does Swiggy have?",
    "What is Swiggy's vision or mission statement?",
    "What was Swiggy's marketing spend for FY 2023-24?",
]

TEST_GROUND_TRUTH = {
    "What was the total expenditure for FY 2023-24?": [
        "139,474", "139", "expenditure", "expenses"
    ],
    "Who is the Chief Financial Officer of Swiggy?": [
        "rahul", "cfo", "financial officer", "chief financial"
    ],
    "What was the depreciation amount for FY 2023-24?": [
        "depreciation", "amortization", "amortisation"
    ],
    "When was Swiggy incorporated?": [
        "2013", "incorporated", "founded", "bundl"
    ],
    "Who are the independent directors on the board?": [
        "independent", "kripalu", "haribhakti", "suparna"
    ],
    "What was other income for FY 2023-24?": [
        "7,080", "6,443", "other income"
    ],
    "What is Swiggy's Dineout business?": [
        "dineout", "dining", "restaurant", "table"
    ],
    "How many employees does Swiggy have?": [
        "employee", "headcount", "workforce", "staff", "people"
    ],
    "What is Swiggy's vision or mission statement?": [
        "vision", "mission", "purpose", "elevating"
    ],
    "What was Swiggy's marketing spend for FY 2023-24?": [
        "marketing", "advertisement", "promotional", "spend"
    ],
}

test_answers = {}

print("Running unseen test questions...\n")
for q in TEST_QUESTIONS:
    print(f"⏳ {q[:60]}...")
    ans            = ask(q, show_context=False)
    test_answers[q] = ans
    print(f" {ans[:100]}")
    time.sleep(8)

test_results = evaluate_accuracy(test_answers, TEST_GROUND_TRUTH)

Running unseen test questions...

⏳ What was the total expenditure for FY 2023-24?...

══════════════════════════════════════════════════════════════════════
Q: What was the total expenditure for FY 2023-24?
══════════════════════════════════════════════════════════════════════

A: The total expenditure for FY 2023-24 was ` 88,020.29 Million (Page 145) and ` 119,276.85 Million for FY 2024-25 (Page 145), however, the question specifically asks for FY 2023-24.

 The total expenditure for FY 2023-24 was ` 88,020.29 Million (Page 145) and ` 119,276.85 Million for
⏳ Who is the Chief Financial Officer of Swiggy?...

══════════════════════════════════════════════════════════════════════
Q: Who is the Chief Financial Officer of Swiggy?
══════════════════════════════════════════════════════════════════════

A: Rahul Bothra is the Chief Financial Officer of Swiggy (Page 26, 147, 103, 146).

 Rahul Bothra is the Chief Financial Officer of Swiggy (Page 26, 147, 103, 146).
⏳ What was the depreciati

**Gradio**

In [20]:
import gradio as gr

def answer_gradio(query, show_ctx, history):
    if not query.strip():
        return history, ""

    results = retrieve(query)
    ans     = generate_answer(query, results)

    ctx_md = ""
    if show_ctx:
        if not results:
            ctx_md = " No chunks passed similarity threshold."
        else:
            for i, r in enumerate(results[:3], 1):
                pages = ", ".join(str(p) for p in r.chunk.pages)
                ctx_md += (
                    f"### Excerpt {i} · Page {pages} · score `{r.score:.3f}`\n\n"
                    f"> {r.chunk.text[:500].strip()} …\n\n---\n\n"
                )
    else:
        ctx_md = "*Context hidden*"

    history = history + [(query, ans)]
    return history, ctx_md


with gr.Blocks(
    title="Swiggy RAG",
    theme=gr.themes.Soft(primary_hue="orange"),
    css="""
        #title    { text-align: center; }
        #subtitle { text-align: center; color: #888; font-size: 14px; }
        footer    { display: none !important; }
    """
) as demo:

    gr.Markdown("#  Swiggy Annual Report — RAG Q&A", elem_id="title")
    gr.Markdown(
        "Answers grounded **strictly** in FY 2023-24 Annual Report · "
        "No hallucination · Page citations included",
        elem_id="subtitle"
    )

    with gr.Row():
        with gr.Column(scale=3):
            chatbot = gr.Chatbot(height=450, bubble_full_width=False)
            with gr.Row():
                query_box = gr.Textbox(
                    placeholder = "Ask anything about the Swiggy Annual Report...",
                    label = "", lines = 2, scale = 5,
                )
                send_btn = gr.Button("Ask ▶", scale=1, variant="primary")
            with gr.Row():
                clear_btn  = gr.Button(" Clear", scale=1)
                ctx_toggle = gr.Checkbox(label="Show context", value=True, scale=2)

        with gr.Column(scale=2):
            gr.Markdown("###  Retrieved Context")
            ctx_box = gr.Markdown(
                value="*Ask a question to see supporting excerpts...*"
            )

    gr.Markdown("###  Try these")
    gr.Examples(
        examples=[
            ["What was Swiggy's total revenue for FY 2023-24?"],
            ["What was the net loss for the year?"],
            ["What was the Gross Order Value for food delivery?"],
            ["How many cities does Swiggy operate in?"],
            ["What is Instamart and how did it perform?"],
            ["Who are Swiggy's Board of Directors?"],
            ["Who is the CEO of Swiggy?"],
            ["What is Swiggy One membership?"],
            ["What are the key risk factors?"],
            ["What are Swiggy's CSR initiatives?"],
            ["What is Swiggy's stock price today?"],
        ],
        inputs=query_box,
        label="",
    )

    gr.Markdown(
        f" **Chunks:** {len(chunks)} &nbsp;|&nbsp; "
        f" **Model:** `{EMBED_MODEL}` &nbsp;|&nbsp; "
        f"⚡ **LLM:** `{GROQ_MODEL}` &nbsp;|&nbsp; "
        f" **Seen:** {eval_results['accuracy']:.0f}% &nbsp;|&nbsp; "
        f" **Unseen:** {test_results['accuracy']:.0f}% &nbsp;|&nbsp; "
        f" **Hit Rate:** {retrieval_results['hit_rate']:.0f}%"
    )

    state = gr.State([])

    send_btn.click(
        answer_gradio,
        inputs  = [query_box, ctx_toggle, state],
        outputs = [chatbot, ctx_box],
    ).then(lambda: ("", []), outputs=[query_box, state])

    query_box.submit(
        answer_gradio,
        inputs  = [query_box, ctx_toggle, state],
        outputs = [chatbot, ctx_box],
    ).then(lambda: ("", []), outputs=[query_box, state])

    clear_btn.click(
        lambda: ([], "*Cleared*", []),
        outputs=[chatbot, ctx_box, state],
    )

demo.launch(share=True, quiet=True)

/tmp/ipykernel_148/3494885911.py:28: DeprecationWarning: The 'theme' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'theme' to Blocks.launch() instead.
  with gr.Blocks(
/tmp/ipykernel_148/3494885911.py:28: DeprecationWarning: The 'css' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'css' to Blocks.launch() instead.
  with gr.Blocks(
/tmp/ipykernel_148/3494885911.py:47: UserWarning: You have not specified a value for the `type` parameter. Defaulting to the 'tuples' format for chatbot messages, but this is deprecated and will be removed in a future version of Gradio. Please set type='messages' instead, which uses openai-style dictionaries with 'role' and 'content' keys.
  chatbot = gr.Chatbot(height=450, bubble_full_width=False)
/tmp/ipykernel_148/3494885911.py:47: DeprecationWarning: The 'bubble_full_width' parameter will be removed in Gradio 6.0. This parameter no longer has any effect.
  chatbot = gr.

* Running on public URL: https://f9900d7f9765de1098.gradio.live
